# GenAI Fundamentals
### Session 3 — How Generative AI Actually Works

---

**Duration:** 1 hour  
**Audience:** Mixed technical (devs + non-devs)  
**Goal:** Build intuition about LLMs, embeddings, and RAG  

---

## 1. The Big Picture (5 min)

### Traditional AI vs. Generative AI

| Traditional AI/ML | Generative AI |
|:-:|:-:|
| **Classifies** existing data | **Creates** new content |
| "Is this spam?" | "Write me an email" |
| "What's the stock price?" | "Summarize this report" |
| Pattern **recognition** | Pattern **generation** |

### The GenAI Landscape (2024-2025)

```
┌─────────────────────────────────────────────────────────────┐
│                    GENERATIVE AI MODELS                      │
├────────────────────────────┬────────────────────────────────┤
│      CLOSED-SOURCE         │         OPEN-SOURCE            │
│                            │                                │
│  ┌──────────────────┐     │    ┌──────────────────┐        │
│  │ OpenAI           │     │    │ Meta             │        │
│  │ GPT-4, GPT-4o    │     │    │ Llama 3          │        │
│  └──────────────────┘     │    └──────────────────┘        │
│  ┌──────────────────┐     │    ┌──────────────────┐        │
│  │ Anthropic        │     │    │ Mistral          │        │
│  │ Claude 4         │     │    │ Mixtral, Large   │        │
│  └──────────────────┘     │    └──────────────────┘        │
│  ┌──────────────────┐     │    ┌──────────────────┐        │
│  │ Google           │     │    │ Others           │        │
│  │ Gemini 2.0       │     │    │ Qwen, Phi, etc.  │        │
│  └──────────────────┘     │    └──────────────────┘        │
├────────────────────────────┴────────────────────────────────┤
│  Key difference: Open-source = you can run it yourself      │
│  Closed-source = use via API, provider controls it          │
└─────────────────────────────────────────────────────────────┘
```

### Why Now? Three Breakthroughs Converged

```
2017                    2020                    2022-2024
 │                       │                       │
 ▼                       ▼                       ▼
┌──────────────┐   ┌──────────────┐   ┌──────────────────┐
│ Transformers │   │ Scaling Laws │   │ RLHF             │
│              │   │              │   │                  │
│ "Attention   │   │ More data +  │   │ Human feedback   │
│  is all you  │──►│ more compute │──►│ makes models     │
│  need"       │   │ = better     │   │ helpful & safe   │
│  (Google)    │   │ models       │   │                  │
└──────────────┘   └──────────────┘   └──────────────────┘
```

---
## 2. How LLMs Work (15 min)

### Tokens: The Atoms of Language

LLMs don't read words. They read **tokens** — pieces of words converted to numbers.

```
┌─────────────────────────────────────────────────────────┐
│  Input text: "The cat sat on the mat"                    │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  Step 1: Split into tokens                              │
│  ┌─────┐ ┌─────┐ ┌─────┐ ┌────┐ ┌─────┐ ┌─────┐      │
│  │ The │ │ cat │ │ sat │ │ on │ │ the │ │ mat │      │
│  └──┬──┘ └──┬──┘ └──┬──┘ └─┬──┘ └──┬──┘ └──┬──┘      │
│     │       │       │      │       │       │          │
│  Step 2: Convert to numbers (IDs)                       │
│     ▼       ▼       ▼      ▼       ▼       ▼          │
│  ┌─────┐ ┌─────┐ ┌─────┐ ┌────┐ ┌─────┐ ┌─────┐      │
│  │ 464 │ │3797 │ │3332 │ │319 │ │ 262 │ │2603 │      │
│  └─────┘ └─────┘ └─────┘ └────┘ └─────┘ └─────┘      │
│                                                         │
│  The model ONLY sees these numbers!                     │
└─────────────────────────────────────────────────────────┘
```

**Surprising tokenization:**
```
"Unbelievable!" → ["Un", "believ", "able", "!"]  (4 tokens)
"AI"            → ["AI"]                          (1 token)
"anthropic"     → ["anthrop", "ic"]               (2 tokens)
```

> **Rule of thumb:** 1 token ≈ 4 characters ≈ 3/4 of a word

### The Prediction Machine

An LLM does **ONE thing**: predict the most likely next token.

```
Input: "The cat sat on the"
                              ┌──────────────────────┐
                              │   Probability of     │
                              │   next token:        │
                              │                      │
                              │   "mat"    → 32%     │
                              │   "floor"  → 18%     │
                              │   "couch"  → 12%     │
                              │   "bed"    →  8%     │
                              │   "table"  →  6%     │
                              │   ...other → 24%     │
                              └──────────────────────┘
```

**How it generates a full response:**
```
Prompt: "What is Python?"

Step 1: "What is Python?" → predicts "Python"
Step 2: "What is Python? Python" → predicts "is"
Step 3: "What is Python? Python is" → predicts "a"
Step 4: "What is Python? Python is a" → predicts "programming"
Step 5: "What is Python? Python is a programming" → predicts "language"
  ...one token at a time until done
```

> **Key insight:** There's no "understanding" — just very sophisticated pattern matching trained on billions of examples.

### Training: How Models Learn

```
┌──────────────────────────────────────────────────────────────────┐
│                     TRAINING PIPELINE                             │
├──────────────────┬───────────────────┬───────────────────────────┤
│                  │                   │                           │
│  PRE-TRAINING    │   FINE-TUNING     │   RLHF                   │
│                  │                   │                           │
│  📚 Read the     │   🎯 Learn to be  │   👍 Learn what humans   │
│  entire internet │   a helpful       │   actually prefer        │
│                  │   assistant       │                           │
│  Data:           │   Data:           │   Data:                   │
│  Books, web,     │   Question/answer │   Human rankings of      │
│  Wikipedia,      │   pairs, instruct │   "response A vs B"      │
│  code repos      │   examples        │                           │
│                  │                   │                           │
│  Result:         │   Result:         │   Result:                 │
│  Knows language  │   Can follow      │   Helpful, harmless,     │
│  & facts         │   instructions    │   honest                  │
│                  │                   │                           │
│  Cost: $$$$$     │   Cost: $$$       │   Cost: $$                │
└──────────────────┴───────────────────┴───────────────────────────┘
```

### Context Window = Working Memory

The context window is how much text the model can "see" at once.

```
┌─────────────────────────────────────────────────────────────┐
│           CONTEXT WINDOW (e.g., 128,000 tokens)             │
│                                                             │
│  ┌────────────┐ ┌─────────────────────┐ ┌───────────────┐  │
│  │  System    │ │   Your conversation  │ │   Model's     │  │
│  │  Prompt    │ │   history so far     │ │   response    │  │
│  │  (rules)   │ │   (all messages)     │ │   (being      │  │
│  │            │ │                     │ │    generated)  │  │
│  └────────────┘ └─────────────────────┘ └───────────────┘  │
│                                                             │
│  ◄──────── EVERYTHING must fit in here ────────────────►    │
└─────────────────────────────────────────────────────────────┘

Model context sizes:
┌─────────────────────────────────────────────┐
│  GPT-4         │  128K tokens (~96K words)   │
│  Claude        │  200K tokens (~150K words)  │
│  Gemini        │  1M+ tokens (~750K words)   │
└─────────────────────────────────────────────┘
```

> **Analogy:** The context window is like a whiteboard. Everything the model is "thinking about" must fit on the whiteboard. Once it's full, old stuff gets erased.

### Temperature: Creativity Dial

Temperature controls randomness in token selection.

```
Temperature = 0 (Precise)          Temperature = 1.0 (Creative)
─────────────────────────          ─────────────────────────────

Probability:                       Probability:

  90% ████████████████████            25% █████████
   5% ██                              22% ████████
   3% █                               20% ███████
   2% █                               18% ██████
                                      15% █████

→ Almost always picks "mat"        → Could pick any of the top options


USE CASES:
┌────────────────────────────────────────────────────────┐
│  Temp 0.0  │  Factual Q&A, code, math                  │
│  Temp 0.5  │  Balanced (most chat applications)         │
│  Temp 0.7  │  Creative writing, brainstorming          │
│  Temp 1.0+ │  Poetry, wild ideas (may be incoherent)   │
└────────────────────────────────────────────────────────┘
```

---
## 3. Limitations, Risks & Practical Realities (7 min)

### The Three Technical Limitations

```
┌─────────────────────────────────────────────────────────────────┐
│                    LLM LIMITATIONS                               │
├─────────────────────┬─────────────────────┬─────────────────────┤
│                     │                     │                     │
│   HALLUCINATIONS    │   KNOWLEDGE CUTOFF  │   NO PRIVATE DATA   │
│                     │                     │                     │
│   The model         │   Training data     │   Can't access      │
│   confidently       │   has a date.       │   YOUR company's    │
│   states things     │   Nothing after     │   documents, DBs,   │
│   that are FALSE.   │   that exists.      │   or internal       │
│                     │                     │   knowledge.        │
│   Example:          │   Example:          │                     │
│   "The Eiffel Tower │   "Who won the      │   Example:          │
│   is 500m tall"     │   2026 World Cup?"  │   "What's our       │
│   (it's 330m)       │   "I don't have     │   refund policy?"   │
│                     │   that info"         │   "I don't know"    │
│                     │                     │                     │
└─────────────────────┴─────────────────────┴─────────────────────┘
```

### Responsible AI Considerations

```
┌─────────────────────────────────────────────────────────────────┐
│                  RISKS TO BE AWARE OF                            │
├─────────────────────┬─────────────────────┬─────────────────────┤
│                     │                     │                     │
│   BIAS IN DATA      │   DATA PRIVACY      │   HUMAN OVERSIGHT   │
│                     │                     │                     │
│   Models learn from │   Sending data to   │   AI assists,       │
│   internet text     │   an API = sending  │   humans decide.    │
│   which contains    │   it to a third     │                     │
│   societal biases.  │   party.            │   Critical decisions│
│                     │                     │   (hiring, medical, │
│   Outputs can       │   Ask: Is this      │   legal) MUST have  │
│   reflect &         │   sensitive? Is it  │   human review.     │
│   amplify these.    │   compliant?        │                     │
│                     │                     │                     │
└─────────────────────┴─────────────────────┴─────────────────────┘
```

**Before deploying AI, ask:**
- What data am I sending to this API? Is it sensitive/confidential?
- Could bias in outputs affect people unfairly (hiring, lending)?
- Who reviews the AI's output before it reaches end users?

### Cost Awareness

```
┌────────────────────────────────────────────────────┐
│  Operation              │  Approximate Cost        │
├─────────────────────────┼──────────────────────────┤
│  GPT-4 / Claude input   │  ~$2-5 per 1M tokens    │
│  Embedding 1M tokens    │  ~$0.10                  │
│  Vector DB storage      │  Pennies per 1K docs     │
│  Fine-tuning            │  $100s-$1000s per run    │
└────────────────────────────────────────────────────┘

A chatbot handling 10K queries/day can cost $100+/day.
Design and token efficiency matter at scale!
```

                    ▼ THE SOLUTION TO THE KNOWLEDGE GAP ▼

        ╔═══════════════════════════════════════════════╗
        ║  RAG: Give the model YOUR data at query time  ║
        ║  so it doesn't rely on memory alone           ║
        ╚═══════════════════════════════════════════════╝

---
## 4. Embeddings & Vector Search (10 min)

### What Are Embeddings?

An embedding converts text into a **list of numbers** that captures its meaning.

```
Text                     Embedding (simplified — usually 768-1536 numbers)
─────────────────────    ─────────────────────────────────────────────────
"king"              →    [0.21, 0.83, -0.45, 0.67, 0.12, ...]
"queen"             →    [0.19, 0.81, -0.42, 0.71, 0.14, ...]  ← similar!
"banana"            →    [0.92, -0.31, 0.55, -0.12, 0.88, ...]  ← different!
```

> **Key insight:** Similar meanings → similar numbers.  
> "King" and "queen" produce similar vectors. "Banana" is far away.

### Semantic Space — Words Have Positions in "Meaning Space"

Imagine plotting text by meaning (reduced to 2D for visualization):

```
                    SEMANTIC SPACE
    ─────────────────────────────────────────
    │
    │          ● "happy"
    │       ● "joyful"       ● "excited"
    │         ● "cheerful"
    │
    │
    │                              ● "sad"
    │                           ● "unhappy"
    │                         ● "depressed"
    │
    │
    │  ● "python"
    │    ● "javascript"
    │      ● "coding"
    │
    │         ● "car"
    │       ● "truck"
    │          ● "vehicle"
    │
    ─────────────────────────────────────────

    Similar concepts CLUSTER TOGETHER
    regardless of the actual words used!
```

This is why vector search can find "password reset help" when you search for "forgot my login" — they're near each other in meaning space!

### Cosine Similarity — Measuring "How Similar?"

Think of embeddings as arrows. Cosine similarity measures the **angle** between them.

```
         Same direction         Perpendicular          Opposite
         = Very similar         = Unrelated            = Opposite meaning

              ↗                      ↑                      ↗
            ↗                    →                        ↙

         Score ≈ 1.0            Score ≈ 0.0            Score ≈ -1.0
```

### Real-World Similarity Scores

| Sentence A | Sentence B | Score |
|:-----------|:-----------|:-----:|
| "How do I reset my password?" | "I forgot my login credentials" | **0.92** |
| "How do I reset my password?" | "Steps to recover account access" | **0.87** |
| "How do I reset my password?" | "What's the weather today?" | **0.13** |
| "The cat sat on the mat" | "A feline rested on the rug" | **0.89** |
| "The cat sat on the mat" | "Stock prices rose sharply" | **0.05** |

> **This is the magic of embeddings:** They understand meaning, not just keywords!

### Vector Databases — The Semantic Search Engine

A vector database stores millions of embeddings and finds the most similar ones **fast**.

```
Traditional keyword search:          Vector/semantic search:
─────────────────────────────        ─────────────────────────────
Query: "password reset"              Query: "I can't log in"

Looks for exact words                Finds MEANING matches
"password" AND "reset"               even with different words

❌ Misses: "account recovery"        ✅ Finds: "password reset guide"
❌ Misses: "login help"              ✅ Finds: "account recovery steps"
❌ Misses: "credentials forgotten"   ✅ Finds: "login troubleshooting"
```

**Popular vector databases:** Pinecone, Weaviate, ChromaDB, pgvector, FAISS

---
## 5. Chunking & Document Processing (8 min)

### Why Chunk?

You can't embed an entire 100-page document as one vector — the meaning gets too diluted.

```
PROBLEM:
┌─────────────────────────────────────────────────────────────┐
│                   100-PAGE DOCUMENT                          │
│                                                             │
│   One embedding for ALL of this?                            │
│   → Meaning is too vague                                    │
│   → Can't pinpoint which PART is relevant                   │
│   → Too big to fit in LLM context anyway                    │
└─────────────────────────────────────────────────────────────┘

SOLUTION:
┌─────────┐ ┌─────────┐ ┌─────────┐ ┌─────────┐ ┌─────────┐
│ Chunk 1 │ │ Chunk 2 │ │ Chunk 3 │ │ Chunk 4 │ │ Chunk 5 │
│ ~500    │ │ ~500    │ │ ~500    │ │ ~500    │ │ ~500    │
│ tokens  │ │ tokens  │ │ tokens  │ │ tokens  │ │ tokens  │
│         │ │         │ │         │ │         │ │         │
│ Own     │ │ Own     │ │ Own     │ │ Own     │ │ Own     │
│embedding│ │embedding│ │embedding│ │embedding│ │embedding│
└─────────┘ └─────────┘ └─────────┘ └─────────┘ └─────────┘

Each chunk:
  ✅ Has focused meaning
  ✅ Can be retrieved independently  
  ✅ Fits in LLM context window
```

### Chunking Strategies

```
STRATEGY 1: Fixed-size chunks
────────────────────────────────────────────
Split every N tokens regardless of content.

[  500 tokens  ][  500 tokens  ][  500 tokens  ]
                ↑
    Might split mid-sentence!

✅ Simple    ❌ Can break meaning


STRATEGY 2: Sentence/paragraph-based
────────────────────────────────────────────
Split at natural boundaries (sentences, paragraphs, sections).

[Paragraph 1][  Paragraph 2  ][Para 3][Paragraph 4  ]

✅ Respects meaning    ❌ Uneven sizes


STRATEGY 3: Semantic chunking
────────────────────────────────────────────
Split when the TOPIC changes (detected by embedding shifts).

[ Topic A content ][ Topic B content ][ Topic C ]

✅ Best quality    ❌ More complex to implement
```

### Overlap: Don't Lose Context at Boundaries

```
WITHOUT overlap — information gets split:
───────────────────────────────────────────────

...the refund policy requires │ customers to submit within 30 days...
           Chunk 1 ends here ─┘└─ Chunk 2 starts here

Neither chunk has the FULL refund policy!


WITH overlap (shared content between chunks):
───────────────────────────────────────────────

Chunk 1: ...the refund policy requires customers to submit within 30 days...
Chunk 2:         ...policy requires customers to submit within 30 days of purchase...
                  ↑─────────── Overlapping region ───────────↑

Both chunks have the complete information!


Typical overlap: 10-20% of chunk size
e.g., 500-token chunks with 50-100 token overlap
```

---
## 6. RAG End-to-End (10 min)

### What is RAG?

**R**etrieval-**A**ugmented **G**eneration

> Give the LLM relevant information BEFORE asking it to answer.

```
WITHOUT RAG:                        WITH RAG:
─────────────────                   ─────────────────

You: "What's our                    You: "What's our
      refund policy?"                     refund policy?"
                                              │
                                              ▼
                                    [Search your docs]
                                              │
                                              ▼
                                    [Found: policy.pdf,
                                     section 4.2]
                                              │
                                              ▼
LLM: "I don't have                  LLM: "Based on your policy
      that information"                   doc, refunds are available
                                          within 30 days with
                                          receipt. See section 4.2"
```

> **Analogy:** It's like an open-book exam. Instead of relying on memory, you give the student the textbook pages they need — THEN ask the question.

### The Full RAG Pipeline

```
═══════════════════════════════════════════════════════════════════
  PHASE 1: INGESTION (one-time setup)
═══════════════════════════════════════════════════════════════════

┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│          │     │          │     │          │     │          │
│  Your    │────►│  Chunk   │────►│  Embed   │────►│  Store   │
│Documents │     │  into    │     │  each    │     │  in      │
│(PDF,docs,│     │  pieces  │     │  chunk   │     │  Vector  │
│ wiki)    │     │          │     │          │     │  DB      │
│          │     │          │     │          │     │          │
└──────────┘     └──────────┘     └──────────┘     └──────────┘


═══════════════════════════════════════════════════════════════════
  PHASE 2: QUERY (every time a user asks a question)
═══════════════════════════════════════════════════════════════════

┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│          │     │          │     │          │     │          │
│  User    │────►│  Embed   │────►│  Search  │────►│ Retrieve │
│ Question │     │  the     │     │  Vector  │     │  Top 3-5 │
│          │     │  query   │     │  DB      │     │  chunks  │
│          │     │          │     │          │     │          │
└──────────┘     └──────────┘     └──────────┘     └─────┬────┘
                                                         │
                                                         ▼
                                  ┌──────────────────────────────┐
                                  │                              │
                                  │  Send to LLM:                │
                                  │  "Using this context:        │
                                  │   [chunk 1] [chunk 2]...     │
                                  │   Answer: {user question}"   │
                                  │                              │
                                  └──────────────┬───────────────┘
                                                 │
                                                 ▼
                                  ┌──────────────────────────────┐
                                  │  LLM generates answer using  │
                                  │  YOUR documents as source    │
                                  └──────────────────────────────┘
```

### RAG in Pseudocode

Here's what the code looks like conceptually (we'll build this for real in future sessions):

```python
# ═══ PHASE 1: INGESTION (run once) ═══

# 1. Load your documents
documents = load_documents("company_docs/")

# 2. Split into chunks
chunks = split_into_chunks(documents, chunk_size=500, overlap=50)

# 3. Create embeddings for each chunk
embeddings = embed(chunks)  # Each chunk → list of numbers

# 4. Store in vector database
vector_db.store(chunks, embeddings)


# ═══ PHASE 2: QUERY (run every time) ═══

# 1. User asks a question
question = "What is our refund policy?"

# 2. Embed the question (same method as documents)
question_embedding = embed(question)

# 3. Find most similar chunks
relevant_chunks = vector_db.search(question_embedding, top_k=3)

# 4. Send to LLM with context
prompt = f"""
Using ONLY the following context, answer the question.
If the answer isn't in the context, say "I don't know."

Context:
{relevant_chunks}

Question: {question}
"""

answer = llm.generate(prompt)
# → "Our refund policy allows returns within 30 days with receipt..."
```

> **Preview:** In Session 4+, we'll implement this for real with Python!

### Why RAG Instead of Fine-tuning?

```
┌──────────────────┬─────────────────────┬─────────────────────┐
│                  │        RAG          │    FINE-TUNING      │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Data freshness   │ ✅ Always current    │ ❌ Frozen at train  │
│                  │ (update docs anytime)│    time             │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Cost             │ ✅ Cheap             │ ❌ Expensive GPU    │
│                  │ (storage + search)   │    training         │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Hallucination    │ ✅ Can cite sources  │ ❌ May still make   │
│                  │                     │    things up        │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Setup time       │ ✅ Hours             │ ❌ Days to weeks    │
├──────────────────┼─────────────────────┼─────────────────────┤
│ Best for         │ Facts, docs,        │ Style, tone,        │
│                  │ knowledge bases     │ specific behavior   │
└──────────────────┴─────────────────────┴─────────────────────┘

TL;DR: Use RAG for knowledge. Use fine-tuning for behavior.
       (Most use cases → RAG)
```

---
## 7. Prompt Engineering Essentials (5 min)

### System Prompts: Setting the Stage

The system prompt tells the model WHO it is and HOW to behave.

```
┌─ System Prompt ─────────────────────────────────────────────┐
│                                                             │
│  You are a helpful customer service agent for Acme Corp.    │
│  - Be concise and professional                              │
│  - Always cite the relevant policy number                   │
│  - If you don't know the answer, say "I don't know"         │
│  - Never make up information                                │
│                                                             │
└─────────────────────────────────────────────────────────────┘
         │
         ▼ This shapes ALL responses in the conversation
```

### Few-Shot: Teaching by Example

Show the model what you want by giving it examples:

```
PROMPT:
────────────────────────────────────────────
Convert informal text to professional email language.

Input: "Hey, can u fix this ASAP?"
Output: "Hello, could you please address this at your earliest convenience?"

Input: "thx for the help!"
Output: "Thank you for your assistance."

Input: "gonna need that report by tmrw"
Output: ???
────────────────────────────────────────────

MODEL RESPONSE:
"I will need that report by tomorrow."

→ It learned the pattern from just 2 examples!
```

### Chain-of-Thought: "Think Step by Step"

```
WITHOUT chain-of-thought:                WITH chain-of-thought:
─────────────────────────                ─────────────────────────

Q: "A store has 3 shirts at            Q: "Think step by step.
    $25 each, with 20% off.                A store has 3 shirts at
    I have $60. Can I buy                  $25 each, with 20% off.
    all 3?"                                I have $60. Can I buy
                                           all 3?"

A: "Yes" ← sometimes wrong!            A: "Step 1: Price per shirt = $25
                                           Step 2: 20% discount = $5 off
                                           Step 3: Sale price = $20 each
                                           Step 4: 3 × $20 = $60
                                           Step 5: $60 = $60 budget
                                           Yes! Exactly enough." ✅
```

> Adding "think step by step" forces the model to show its work,  
> which dramatically improves accuracy on reasoning tasks.

---
## Recap & What's Next

```
┌─────────────────────────────────────────────────────────────────┐
│                     WHAT WE COVERED TODAY                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ● LLMs = next-token prediction machines                        │
│  ● Tokens = how text becomes numbers                            │
│  ● Context window = model's working memory                      │
│  ● Embeddings = "meaning fingerprints" for text                 │
│  ● Vector search = find by meaning, not keywords                │
│  ● Chunking = breaking docs into searchable pieces              │
│  ● RAG = retrieve relevant docs → feed to LLM → generate       │
│  ● Prompt engineering = system prompts, few-shot, CoT           │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│                      COMING UP NEXT                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Session 4+: Build a working RAG pipeline in Python             │
│  → Calling LLM APIs                                             │
│  → Generating embeddings                                        │
│  → Setting up a vector database                                 │
│  → Building an end-to-end Q&A system                            │
│  → Agents & tool use                                            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Quick Reference Card

| Term | Plain English |
|------|---------------|
| **Token** | A piece of a word (~4 characters) |
| **LLM** | Large Language Model — predicts next token |
| **Context Window** | How much text model can see at once |
| **Temperature** | Creativity dial (0=precise, 1=creative) |
| **Embedding** | Text converted to numbers capturing meaning |
| **Vector** | A list of numbers (the embedding itself) |
| **Cosine Similarity** | Score measuring how similar two embeddings are |
| **Vector DB** | Database optimized for similarity search |
| **Chunking** | Splitting documents into smaller pieces |
| **RAG** | Retrieve relevant docs, then generate answer |
| **System Prompt** | Instructions that shape model behavior |
| **Few-shot** | Teaching by giving examples in the prompt |
| **Chain-of-Thought** | Asking model to reason step by step |